<div style="background-color:#007033; color:white; padding:18px 22px; border-radius:6px; margin-bottom:10px;"><h1 style="color:white; margin:0 0 4px 0;">Week 1 — Introduction to Production AI Systems</h1><div style="font-size:15px; opacity:0.9;">FraudGuard Case Study · Stage 1 of 10 · INFT 41000, Omar Altrad, Durham College</div></div>

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">This week</b><br>Experimentation vs. production AI. You'll build a quick fraud-detection model, watch it look great and quietly fail, then map out what production would actually require — and start fixing the first gap you find with real version control (git + GitHub).<br><br><b>Case study:</b> starting today, every week extends one running project — <b>FraudGuard</b>, a fraud detector for a synthetic mobile-money platform. Synthetic data on purpose — no real financial data belongs in a classroom.</div>

| # | Section |
|---|---|
| 0 | Setup |
| 1 | Generate the dataset |
| 2 | Explore it |
| 3 | Baseline model |
| 4 | Production gap analysis |
| 5 | Fixing Gap #1: version control with Git and GitHub |
| 6 | Save your work |

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">One-time project setup</b><br>This notebook lives inside the shared <b>Course_Project</b> folder (one <code>.venv</code>, <code>data/</code>, <code>models/</code>, <code>notebooks/</code>, <code>src/</code> for the whole course). <b>First time only:</b> follow <code>README.md</code> at the project root (open the <b>Course_Project</b> folder in VS Code, create <code>.venv</code>, install <code>requirements.txt</code>, install the Python + Jupyter extensions). Already set up? Just open this notebook, pick the <code>.venv</code> kernel, and run the cell below.</div>

<div style="background-color:#FDE8E6; border-left:5px solid #B22222; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B22222;">If something breaks</b><br>Full project-setup troubleshooting table (kernel picker, path errors, PowerShell activation, etc.) is in <code>README.md</code> at the project root — check there first.</div>

### Experimentation vs. production

| | Notebook | Production |
|---|---|---|
| Reproducibility | "ran on my machine" | pinned deps, versioned data & model |
| Reliability | runs once, for you | runs correctly, unattended, 24/7 |
| Monitoring | you eyeball it | automated metrics + alerts |
| Security / legal | not a concern | auth, encryption, fairness, audit trail |
| Change | rerun the cell | CI/CD, rollback, incident response |

That's the whole roadmap of this course — Weeks 2–10 each turn one row into a real skill.

**Run this first, every notebook, every week** — it finds the project root and sets up `DATA_DIR`, `MODELS_DIR`, and imports from `src/`.

In [ ]:
import sys
from pathlib import Path

def find_project_root():
    '''Walk up from the current working directory until we find the
    Course_Project root (identified by having both requirements.txt
    and a src/ folder). Works no matter which VS Code cwd setting is in
    effect for notebooks (fileDirname or workspaceFolder), because either
    way this notebook lives somewhere inside the project tree.'''
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Course_Project root (a folder containing both "
        "requirements.txt and src/). Open this notebook from inside the project "
        "folder in VS Code (File > Open Folder > Course_Project), or set PROJECT_ROOT "
        "manually, e.g. PROJECT_ROOT = Path(r'C:\path\to\INFT41000\Course_Project')."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"

for required_dir in (DATA_DIR, MODELS_DIR):
    if not required_dir.is_dir():
        raise FileNotFoundError(
            f"Missing folder: {required_dir}\n"
            "Create it first (see the folder-structure instructions in 00_Setup.ipynb) -- "
            "this notebook does not create folders for you."
        )

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4.5)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Setup OK")

## 1. Generate the FraudGuard Dataset

A PaySim-style synthetic mobile-money dataset, built by `src/fraudguard_datagen.py` — shared
by every week's notebook via the project setup cell above (no copying needed).

| Column | Meaning |
|---|---|
| `type` | CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER |
| `amount`, `*balance*` | transaction amount, origin/destination balances before & after |
| `isFraud` | label — 1 if fraudulent |
| `isFlaggedFraud` | 1 if a simple legacy rule would have caught it (deliberately imperfect) |

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Why the fraud rate looks high</b><br>Real PaySim fraud is ~0.13% of transactions — too rare for a small teaching dataset. We use ~0.6% here; we come back to realistic extreme imbalance in Weeks 7 & 8.</div>

In [ ]:
from fraudguard_datagen import generate_transactions

df = generate_transactions(n_rows=200_000, fraud_rate=0.006, missing_rate=0.001, seed=RANDOM_SEED)
df.to_csv(DATA_DIR / "fraudguard_transactions_week1.csv", index=False)
print(df.shape)
df.head()

## 2. Explore the Data

In [ ]:
fraud_rate_overall = df["isFraud"].mean()
print(f"Overall fraud rate: {fraud_rate_overall:.4%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
df["isFraud"].value_counts().plot(kind="bar", ax=axes[0], color=["#4C72B0", "#C44E52"])
axes[0].set_title("Class balance: isFraud")
axes[0].set_xticklabels(["Legitimate (0)", "Fraud (1)"], rotation=0)

df["type"].value_counts().plot(kind="bar", ax=axes[1], color="#4C72B0")
axes[1].set_title("Transaction type volume")
plt.tight_layout()
plt.show()

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Exercise 1.1</b><br>Compute the fraud rate <b>per transaction type</b>, sorted descending. Which types have zero fraud, and why might a fraud model skip scoring those in production?</div>

In [ ]:
# YOUR CODE HERE — fraud rate by type, sorted descending
fraud_by_type = None
raise NotImplementedError
print(fraud_by_type)

*Your answer:*

## 3. Baseline Model — "It Works On My Machine"

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

features = ["amount", "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
model_df = df.dropna(subset=features).copy()

X, y = model_df[features], model_df["isFraud"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4%}")

<div style="background-color:#FDE8E6; border-left:5px solid #B22222; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B22222;">The accuracy paradox</b><br>99%+ accuracy — but ~0.6% fraud means predicting "not fraud" for everything would already score ~99.4%. Accuracy alone tells us almost nothing here.</div>

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Exercise 1.2</b><br>Print the confusion matrix and a classification report for the <b>fraud class</b> (precision/recall/F1). Then answer below: at millions-of-transactions/day scale, what does a false negative cost vs. a false positive, and which would you prioritize reducing?</div>

In [ ]:
# YOUR CODE HERE — confusion matrix + classification_report
from sklearn.metrics import confusion_matrix, classification_report

raise NotImplementedError

*Your answer:*

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Exercise 1.3 — Production Readiness Gap Table</b><br>Your manager says "ship it tomorrow." For each gap below, note its category (Business / Technical / Operational) and one sentence on the risk if ignored. Add two gaps of your own.</div>

| # | Gap | Category | Risk if ignored |
|---|---|---|---|
| 1 | No model/data versioning | | |
| 2 | No input validation | | |
| 3 | No monitoring | | |
| 4 | Hardcoded config | | |
| 5 | No access control | | |
| 6 | No latency testing | | |
| 7 | No documentation | | |
| 8 | *(your own)* | | |
| 9 | *(your own)* | | |

## 5. Fixing Gap #1: Version Control with Git and GitHub

Gap #1 in the table above was **no model/data versioning**. That's actually two separate
problems — versioning *data and models* (a bigger topic, tackled properly with MLflow in
Week 9) and versioning *code* (this notebook, `src/fraudguard_datagen.py`, everything you
write all course). The second one you can start fixing right now, in the next 15 minutes,
with the single most-used tool in professional software development: **git**.

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Key terms</b><br><b>git</b> — a version control system: it snapshots your project's history so you can see what changed, when, and by whom, and roll back if something breaks.<br><b>repository (repo)</b> — a project folder git is tracking.<br><b>commit</b> — a saved snapshot with a message describing what changed.<br><b>branch</b> — an independent line of work; lets you experiment without touching the working version.<br><b>GitHub</b> — a hosted service for git repos, adding collaboration features (pull requests, issues, code review) git itself doesn't have. Git works with no internet connection at all; GitHub is where you optionally back up and share that history.</div>

### 5.1 One-time setup (skip if you've used git before)

This is the ONE command-line step in this whole section — everything else below happens
through clicks in VS Code's **Source Control panel** (the icon in the left sidebar that
looks like branching lines, or press `Ctrl+Shift+G`).

If you've never used git on this machine, open the integrated terminal (`` Ctrl+` ``) and
set your identity once — this is global, not per-project, so you only do it once ever:

```bash
git config --global user.name "Your Name"
git config --global user.email "omar.al-trad@durhamcollege.ca"
```

If a commit is ever rejected with something like `Please tell me who you are`, the two `git config` commands above haven't run yet — open the terminal, run them, then try again. Otherwise, you're ready for 5.2 below.

### 5.2 Initialize the repo and make your first commit

One repo for the whole `Course_Project` folder — not one per week. `data/` and `models/`
get **excluded** on purpose: they're large, regenerable, and belong to a different kind of
versioning (data/model versioning — Week 9's job, with MLflow). Code, notebooks, and config
are what git tracks well.

**Do this now, step by step:**

1. **Create `.gitignore`** — in the Explorer panel, right-click `Course_Project` → **New File** →
   name it `.gitignore` (the leading dot matters) → paste in:
   ```
   .venv/
   __pycache__/
   *.pyc
   .ipynb_checkpoints/
   data/
   models/
   api/
   loadtest/
   .env
   ```
   Save it (`Ctrl+S`).

2. **Open the Source Control panel** (`Ctrl+Shift+G`). If this folder isn't a git repo yet,
   you'll see one button: **Initialize Repository** — click it. (That's the visual equivalent
   of `git init`.)

3. Under **Changes**, you'll see every file in the project except the ones `.gitignore` just
   excluded. Stage only the project skeleton for this first commit: select `.gitignore`,
   `requirements.txt`, `README.md`, `src/`, and `00_Setup.ipynb` (`Ctrl`+click to multi-select),
   then click the **+** that appears next to them (or right-click → **Stage Changes**).

4. Type a commit message in the box at the top of the panel: `Initial commit: project
   skeleton, requirements, shared datagen module`. Click the checkmark **✓** (or `Ctrl+Enter`)
   to commit.

5. **Make sure the branch is named `main`**: look at the bottom-left status bar. If it says
   `master` instead, click it, choose **Rename Branch**, and type `main`. (Newer git installs
   already default to `main` — if yours does, skip this.)

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Exercise 1.4</b><br>Do the same thing for <b>this week's own files</b>: in the Source Control panel, stage the `notebooks/Week01_Introduction_to_Production_AI/` folder, then commit with a message that says <i>what</i> changed and <i>why</i> a teammate would care — <code>"add Week 1 notebook"</code> is weak; <code>"Add Week 1: baseline fraud model + production gap analysis"</code> is a real commit message.</div>

**Do this now:** select the Week 1 folder in **Changes**, stage it, type the commit message above, and click the checkmark.

### 5.3 Connect to GitHub

1. On [github.com](https://github.com), click **New repository**. Name it something like
   `fraudguard-course-project`. Leave it **empty** (no README/gitignore/license — you already
   have those locally) and set it to **Private** — your work, not a public template.
2. Back in VS Code's Source Control panel, click the **...** menu → **Push** → **Push to...**
   (or the **Publish Branch** button, if VS Code shows one) and paste in the repository URL
   GitHub just gave you. This does the equivalent of `git remote add origin ...` and
   `git push -u origin main` in one click, and VS Code will prompt you to sign in to GitHub
   the first time.
3. Refresh the GitHub page — your commits should now be there.

From here on, **commit at the end of every week** (stage + commit + push, all in the Source
Control panel) — and by Week 10 you'll have a real commit history documenting the whole
project's evolution, which is itself evidence of good engineering practice.

<div style="background-color:#DBF2FF; border-left:5px solid #005291; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#005291;">Why this step needs you specifically</b><br>Everything above this point works the same for anyone who follows it. Publishing needs YOUR GitHub account's identity — VS Code will pop up a real GitHub sign-in the first time, and only you can complete that. Every later class step reuses this same connection automatically.</div>

### 5.4 Branches, and your first merge conflict

Real teams rarely edit the main line of work directly — they branch, make a change, then
merge back. Merge conflicts happen when two branches change the *same lines* of the *same
file* differently. They're not an error to be afraid of; they're git asking a question only
a human can answer: *which version did you actually mean?*

The steps below set up a real conflict on a throwaway file, and then resolve it using VS
Code's built-in merge editor — buttons right above the conflicting lines, no memorizing
conflict-marker syntax — so you can see exactly what a conflict looks like and how it gets
fixed, before you hit one for real on work that matters.

**Set up the conflict:**

1. Create a new file `git_conflict_lab.txt` (right-click `Course_Project` → **New File**).
   Type `FraudGuard project status: in progress`, save. Stage and commit it (message:
   `Add project status file`).

2. Create a new branch: click the branch name in the bottom-left status bar →
   **Create new branch...** → name it `update-status` → press Enter. VS Code switches you
   onto it automatically.

3. On this branch, edit `git_conflict_lab.txt` to say `FraudGuard project status: Week 1
   complete`, save, stage, and commit (message: `Mark Week 1 complete on update-status
   branch`).

4. Switch back to `main`: click the branch name in the status bar again and select `main`.

5. On `main`, edit `git_conflict_lab.txt` to a *different* line: `FraudGuard project status:
   baseline model built, in review`, save, stage, and commit (message: `Update status on
   main`).

6. Merge the branches: click the **...** menu in the Source Control panel → **Branch** →
   **Merge Branch...** → select `update-status`. VS Code will report a conflict in
   `git_conflict_lab.txt` — that's expected, not a bug.

**Resolving it**: open `git_conflict_lab.txt`. VS Code shows both versions inline with **Accept Current Change**, **Accept Incoming Change**, **Accept Both Changes**, and **Compare Changes** buttons right above the conflicting lines — pick one (or edit the line by hand into a single combined version), then save. The `<<<<<<<`, `=======`, `>>>>>>>` markers disappear once it's resolved.

**Finish it:** back in the Source Control panel, `git_conflict_lab.txt` now shows under **Merge Changes** — stage it, and commit (VS Code will suggest something like `Merge branch 'update-status'`; that's a fine message, or write your own). Then clean up: delete `git_conflict_lab.txt` (right-click it in the Explorer → **Delete**), stage and commit the deletion (message: `Remove git conflict lab file`), and delete the `update-status` branch (Command Palette, `Ctrl+Shift+P` → **Git: Delete Branch** → select `update-status`) — this was practice, not real project content.

<div style="background-color:#FCF3D9; border-left:5px solid #B58800; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#B58800;">Exercise 1.5 — Pull requests (no code, do this on GitHub)</b><br>On GitHub, a <b>pull request (PR)</b> proposes merging one branch into another, with a space for review comments before it happens — the collaboration layer git itself doesn't have. Create one branch, push it, open a PR into <code>main</code> on GitHub's website (even solo, this is how real review starts), and paste the PR's URL in the cell below as a comment. You do not need to actually merge it.</div>

In [ ]:
# Paste your pull request's URL here as a comment once you've created it on GitHub.com:
# https://github.com/YOUR-USERNAME/fraudguard-course-project/pull/1

## 6. Save Your Work — Needed for Week 2

In [ ]:
import joblib

MODEL_PATH = MODELS_DIR / "week1_baseline_model.joblib"
joblib.dump(clf, MODEL_PATH)
print(f"Saved: {MODEL_PATH.resolve()}")
print(f"Saved: {(DATA_DIR / 'fraudguard_transactions_week1.csv').resolve()}")

<div style="background-color:#E0F1E8; border-left:5px solid #007033; padding:10px 14px; border-radius:4px; margin:8px 0;"><b style="color:#007033;">Submit</b><br>This notebook, run top-to-bottom with no errors · Exercises 1.1–1.5 completed · a real GitHub repo with your Week 1 commit history and at least one open pull request · a short reflection (5–8 sentences): why can 99% accuracy be unfit for production, and what could go wrong at real scale if you picked one Exercise 1.3 gap and ignored it?<br><br><b>Next week:</b> Week 2 — formally scoring FraudGuard's production readiness. Keep committing to git at the end of every week from here on.</div>